← [Cambio de dominio](08-cambio-de-dominio.ipynb) · [Índice](00-indice.ipynb) · Siguiente: [Votación y decisión](10-votacion-y-decision.ipynb) →

# 09 · Las otras dos señales

El sistema experto necesita hechos para razonar: transparencia, color, tapa,
forma. Alguien tiene que mirar la foto y producirlos. De eso se encargan las
dos piezas de este documento.



## Estado vigente del proyecto (actualizado el 13 de agosto de 2026)

Esta serie conserva explicaciones y resultados históricos, pero la referencia
operativa actual es la siguiente:

- El sistema experto tiene **193 reglas**, CF estilo MYCIN, meta-reglas,
  encadenamiento hacia adelante y hacia atrás. Su voto usa OpenAI como
  proveedor principal, con heurísticas OpenCV que refinan atributos.
- El modelo local que participa en la decisión es **MobileNetV2 TFLite
  float32**, corrida `run_20260721_2129`; clasifica solo `plastico | vidrio`.
  MobileNetV3-Large INT8 está archivado como respaldo y **no emite votos**.
- En 1.000 capturas OV3660/QVGA, V2 obtuvo **71,60 %** de exactitud y
  **71,25 %** de macro-F1; V3 INT8 obtuvo 57,10 % y 57,09 %. La validación
  histórica de 98,43 % no describe por sí sola el rendimiento del robot.
- La ESP32-CAM toma tres fotos: se suman los seis votos válidos de ambas
  fuentes. `desconocido` es abstención; un empate se resuelve con el proveedor.
  Si el proveedor se abstiene las tres veces, el modelo local necesita 3/3.

La documentación operativa es [`ia/vision-service/README.md`](../../ia/vision-service/README.md),
[`model/README.md`](../../ia/vision-service/model/README.md) y
[`PLAN-INTEGRACION-VISION-MAIN-2026-08-13.md`](../../docs/PLAN-INTEGRACION-VISION-MAIN-2026-08-13.md).


## El proveedor de visión

Un modelo de lenguaje multimodal —OpenAI por defecto, con Claude y Gemini como
alternativas configurables— recibe la foto y devuelve los **nueve atributos**
del vocabulario definido en [03](03-sistema-experto-reglas.ipynb).

El punto importante del diseño:

> El proveedor **no decide el material**. Describe cómo se ve el objeto.

La decisión la toma el sistema experto a partir de esa descripción. Es una
separación deliberada de responsabilidades:

```
proveedor de visión  →  "transparencia alta, tapa corona, color ámbar"
sistema experto      →  "entonces es VIDRIO, por las reglas R02 y R14, CF 0,95"
```

Se podría haber preguntado directamente "¿esto es vidrio o plástico?". No se
hizo, por tres razones:

1. **Explicabilidad.** Si el proveedor decidiera, no habría reglas que citar y
   se perdería el `rule_applied`.
2. **Control.** Las reglas son auditables y editables por el equipo; el
   comportamiento interno del proveedor, no.
3. **Validación.** Los atributos tienen valores cerrados y `validator.py` puede
   rechazar los inválidos ([05](05-motor-de-inferencia.ipynb)). Una respuesta libre
   no se puede validar así.

La configuración vive en `vision/vision_config.py`, que resuelve proveedor y
modelo desde variables de entorno y corrige typos conocidos en los nombres de
modelo —un detalle nacido de que un nombre mal escrito produce un HTTP 404
silencioso.



## Las heurísticas OpenCV

`vision/visual_heuristics.py` (~600 líneas) mide propiedades de la imagen con
**visión por computadora clásica**: histogramas de color, detección de bordes,
análisis de brillo. Sin aprendizaje de ningún tipo — pura aritmética sobre
píxeles, con umbrales escritos a mano.

Es la tercera familia del proyecto, y conviene situarla:

| | Sistema experto | Heurísticas OpenCV | Red neuronal |
| --- | --- | --- | --- |
| Opera sobre | Atributos simbólicos | Píxeles | Píxeles |
| Conocimiento | Reglas escritas | Fórmulas escritas | Aprendido |
| Explicable | Sí | Sí | No |
| Aprende de datos | No | No | Sí |

Su valor es que son **deterministas y gratuitas**: no dependen de una llamada de
red, no tienen cuota, no varían entre ejecuciones. Aportan una medición
independiente sobre las mismas propiedades que el proveedor estima, y sirven
como contraste cuando este falla o no está disponible.

Su límite es el mismo que el del sistema experto: solo captan lo que alguien
pensó en programar.



## Las cuatro piezas juntas

```
        foto de la OV3660 en la ESP32-CAM
                 │
      ┌──────────┴───────────┐
      │                      │
┌─────▼──────────────┐   ┌───▼────────────────┐
│ proveedor de visión│   │  modelo local TFLite│
│ 9 atributos        │   │  (red neuronal)    │
└─────┬──────────────┘   └───┬────────────────┘
      │                      │
┌─────▼──────────────┐       │
│ heurísticas OpenCV │       │
│ medición directa   │       │
└─────┬──────────────┘       │
      │                      │
┌─────▼──────────────┐       │
│ sistema experto    │       │
│ 193 reglas + CF    │       │
└─────┬──────────────┘       │
      │                      │
   VOTO A                 VOTO B
```

Las piezas del camino izquierdo forman **una sola señal**: el proveedor y las heurísticas
alimentan al sistema experto, que emite el veredicto. El modelo local va por su
cuenta, de la foto directo al resultado.

Por eso, con tres fotos por residuo, salen **seis diagnósticos**: tres del
camino simbólico y tres del camino aprendido.



En la integración actual, ese voto independiente lo emite MobileNetV2 TFLite float32.
Su carácter binario obliga a tratarlo como respaldo corroborador, no como una
fuente capaz de reconocer objetos fuera de `plastico | vidrio`.

## Por qué el modelo local no pasa por el sistema experto

Podría haberse conectado su salida como un hecho más (`confianza_ml` ya existe
como atributo). Se decidió mantenerlo separado para que los dos caminos sean
**independientes**.

Si el veredicto de la red neuronal entrara al sistema experto, sus errores
contaminarían el razonamiento simbólico y ya no habría dos opiniones, sino una
sola con pasos extra. Manteniéndolos separados se conserva lo que hace valiosa
la combinación: dos sistemas que fallan por motivos distintos
([02](02-dos-formas-de-saber.ipynb)).

Cómo se combinan esos seis diagnósticos es el siguiente documento.

---

← [Cambio de dominio](08-cambio-de-dominio.ipynb) · [Índice](00-indice.ipynb) · Siguiente: [Votación y decisión](10-votacion-y-decision.ipynb) →
